# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. The dataset is described using a Croissant schema and provides detailed information on protein abundance, modifications, and sequences.

### Dataset Source
This dataset's Croissant schema is available at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` and `pandas` are installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
meta = dataset.metadata

# Print a summary
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Let's list available record sets and their fields, referencing each by their `@id` field.

In [ ]:
# List all record sets and their fields' IDs

record_sets = meta.record_sets
print(f"Found {len(record_sets)} record sets.")

for rs in record_sets:
    print(f"\nRecordSet: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")

## 3. Data Extraction
Load data from each record set into a dataframe. We use the `@id` for the record sets. First, select a usable record set for exploration.

In [ ]:
# Get the list of all record set @id's
record_set_ids = [rs.id for rs in meta.record_sets]
print("Record set @id's available:")
for rs in meta.record_sets:
    print(f"- {rs.name}: {rs.id}")

# For this notebook, let's pick the first available record set
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"\nUsing record set: {selected_record_set_id}")
else:
    selected_record_set_id = None

# Load all records from each record set into dataframes
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show the columns of the selected record set DataFrame
if selected_record_set_id:
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field by its `@id` (from the record set's field list above), filter and transform the data using that field, and group by a categorical field if available.

For this exploration, we will dynamically identify a numeric field and a group field to demonstrate functionality.

In [ ]:
import numpy as np

# Helper: Get a numeric field and a group field from the selected record set
num_field_id = None
group_field_id = None
if selected_record_set_id:
    selected_rs = next(rs for rs in meta.record_sets if rs.id == selected_record_set_id)
    # Find a numeric field (Integer or Float)
    for f in selected_rs.fields:
        if f.data_type in ('schema:Integer', 'schema:Float', 'schema:Number'):
            num_field_id = f.id
            break
    # Find the first non-numeric field as group
    for f in selected_rs.fields:
        if f.data_type == 'schema:Text':
            group_field_id = f.id
            break

print(f"Numeric field selected: {num_field_id}")
print(f"Group field selected: {group_field_id}")

df = dataframes[selected_record_set_id]

# Ensure the numeric field is convertible to float
if num_field_id in df.columns:
    df[num_field_id] = pd.to_numeric(df[num_field_id], errors='coerce')
    threshold = df[num_field_id].quantile(0.75) if df[num_field_id].notnull().any() else 0
    filtered_df = df[df[num_field_id] > threshold].copy()
    print(f"Filtered records with {num_field_id} > {threshold}:")
    display(filtered_df.head())
    
    # Normalization
    if filtered_df[num_field_id].notnull().any():
        filtered_df[f"{num_field_id}_normalized"] = (
            (filtered_df[num_field_id] - filtered_df[num_field_id].mean()) / filtered_df[num_field_id].std()
        )
        print(f"Normalized {num_field_id} for filtered records:")
        display(filtered_df[[num_field_id, f"{num_field_id}_normalized"]].head())
    
    # Grouping, if group field available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[num_field_id].mean().reset_index()
        )
        print(f"Grouped data by {group_field_id}: (mean {num_field_id} per group)")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field, and if grouping was possible, plot the group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for numeric field
if num_field_id in df.columns and df[num_field_id].notnull().any():
    plt.figure(figsize=(7, 4))
    sns.histplot(df[num_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(num_field_id)
    plt.title(f'Distribution of {num_field_id}')
    plt.show()

# If grouping was done, barplot
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_df, x=group_field_id, y=num_field_id)
    plt.xticks(rotation=90)
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {num_field_id}')
    plt.title(f'Mean {num_field_id} by {group_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- We successfully loaded the FAIR² mass spectrometry dataset using its Croissant schema and explored available record sets, fields, and data.
- We identified a numeric field for exploratory data analysis, filtered and normalized relevant records, and visualized the results.
- This approach, referencing all entities by their `@id`, is robust for working with complex and richly described datasets in the Croissant standard.

You can now build on these steps for more advanced feature engineering or machine learning analysis!